# **Implementing a simple retriever to chat with documents**

Flow:
Load Document -> Chunk -> Create Embeddings -> Store Embeddings -> Use Retriever

In [2]:
!pip install pymupdf --q


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.1.13
0.0.29


In [4]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import utils

In [5]:
DOC_NAME = "ACME Corp Employee Leave Policy Handbook.pdf"
VECTOR_DB_PATH = "local_faiss_db"

In [6]:
doc = PyMuPDFLoader(DOC_NAME).load()

In [7]:
doc[0].page_content

'ACME Corp: Employee Leave Policy \nHandbook \n1. Sick Leave \nAll full-time employees accrue sick leave starting from their first day of employment. Employees \nreceive twelve days of paid sick leave per calendar year, which is accrued at a steady rate of \none day per month. Sick leave can be used for personal illness, medical appointments, mental \nhealth days, or caring for an immediate family member who is ill. To utilize this leave, employees \nmust notify their manager via email or Slack by 8:30 AM on the day of their absence. Please \nnote that a formal medical certificate must be submitted to HR for any sick leave absences \nexceeding three consecutive days. \n2. Vacation Leave \nFull-time employees become eligible to take accrued vacation leave after completing their initial \nninety-day probationary period. The company provides fifteen days of paid vacation leave per \ncalendar year. Unused vacation days up to a maximum of five days can be carried over to the \nnext calendar

In [9]:
doc[1].page_content

"date of delivery. A formal request along with a medical certificate stating the expected delivery \ndate must be submitted to HR at least thirty days before the intended start of the leave. Upon \ncompletion of the leave, ACME Corp guarantees that the employee will return to their same \nposition or an equivalent role with the matching salary and benefits. \n5. Adoption and Surrogacy Leave \nThe company supports all paths to parenthood by providing adoption and surrogacy leave to \neligible employees who have completed at least twelve months of continuous service. The \nprimary caregiver is entitled to twelve weeks of fully paid leave to bond with the child, starting \nfrom the official date of placement or the child's birth via surrogacy. A formal application for this \nleave along with official legal documentation from the adoption agency or surrogacy agreement \nmust be submitted to HR at least thirty days prior to the expected leave date. \n6. Paternity and Co-Parent Leave \nTo su

In [8]:
len(doc)

2

In [10]:
# \n\n - Preserves the macro-structure by keeping paragraphs intact.
# \n - Splits large paragraphs into individual lines.
# (?<=\. ) - Uses regex lookbehind to split at sentence boundaries.
#   - Splits by spaces to avoid cutting words in half.
# "" - Hard-cuts exactly at the character limit as a final fallback.

text_splitter  = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)
splitted_text=text_splitter.split_documents(doc)

In [11]:
len(splitted_text)

9

In [12]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [13]:
vectordb = FAISS.from_documents(
    documents=splitted_text,
    embedding=embeddings
)

In [14]:
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)

In [15]:
retriever = vectordb.as_retriever(search_kwargs={"k": 2})

In [16]:
print(retriever.invoke("What is the policy for taking a vacation leave")[0].page_content)

note that a formal medical certificate must be submitted to HR for any sick leave absences 
exceeding three consecutive days. 
2. Vacation Leave 
Full-time employees become eligible to take accrued vacation leave after completing their initial 
ninety-day probationary period. The company provides fifteen days of paid vacation leave per 
calendar year. Unused vacation days up to a maximum of five days can be carried over to the 
next calendar year, after which any remaining unused balance is forfeited. Vacation leave must


In [17]:
from langchain.chains import RetrievalQA

In [18]:
Retriever_chain = RetrievalQA.from_chain_type(llm,
                                              retriever=vectordb.as_retriever(),
                                              return_source_documents=True
                                             )

In [19]:
response = Retriever_chain.invoke("How many days of sick leave do I get?")
print(response.get('result'))

You receive twelve days of paid sick leave per calendar year at ACME Corp.


In [20]:
response = Retriever_chain.invoke("How many days of vacation leave do I get?")
print(response.get('result'))

Full-time employees at ACME Corp are provided with fifteen days of paid vacation leave per calendar year.
